<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Batching y el Shuffling</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Entrenar ejemplo a ejemplo suele ser ineficiente, y entrenar siempre en el mismo orden puede introducir sesgos no deseados. Este notebook se centra en dos ideas operativas clave: el batching y el shuffling.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Ambas decisiones afectan la estabilidad del entrenamiento, el uso de memoria y la forma en que el modelo ve los datos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué aporta trabajar por lotes y por qué mezclar el orden de los ejemplos suele mejorar el proceso.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">El modo en que entregas los datos al modelo también forma parte del diseño experimental.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Batching agrupa ejemplos en minibatches y shuffling altera su orden de presentación entre iteraciones.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación del dataset
"""Este bloque crea un dataset simple utilizando tensores.
Este bloque importa "torch" y "TensorDataset" de "torch.utils.data" para crear un dataset simple.

La variable "datos" almacena un tensor que corresponde a las características de los datos.
La variable "etiquetas" almacena un tensor que corresponde a las etiquetas de los datos.

Aplicamos la clase "TensorDataset()" a la variable "dataset" para crear un dataset que contiene los datos y etiquetas.
Los argumentos "datos" y "etiquetas" son los tensores que contienen los datos y las etiquetas.

La variable "dataset" es una instancia de la clase "TensorDataset()" que contiene los datos y etiquetas proporcionados.
El dataset sirve como fuente de datos para aplicar batching y shuffling."""

import torch
from torch.utils.data import TensorDataset

datos = torch.tensor([[1.0, 2.0],
                      [3.0, 4.0],
                      [5.0, 6.0],
                      [7.0, 8.0]])
etiquetas = torch.tensor([0, 1, 0, 1])

dataset = TensorDataset(datos, etiquetas)

dataset # Muestra una representación del objeto dataset.


In [ ]:
# Bloque 2 — Batching sin shuffling
"""Este bloque crea un dataloader sin shuffling.
Este bloque importa "DataLoader" de "torch.utils.data" para crear un dataloader a partir del dataset.

Aplicamos la clase "DataLoader()" a la variable "dataloader_ordenado" para crear un dataloader a partir del dataset original.
El argumento "dataset" corresponde a la instancia del dataset original que se desea cargar en el dataloader.
El parámetro "batch_size=" especifica el tamaño máximo de los lotes que se obtendrán del dataloader.
El parámetro "shuffle=" indica si se debe mezclar el orden de las muestras antes de crear los lotes.
La variable "dataloader_ordenado" almacena un objeto iterable que entrega los datos originales en lotes sin mezclar.

Creamos un contenedor de lista llamado "lotes_ordenados" para almacenar los lotes obtenidos del dataloader.
La variable "lotes_ordenados" almacena los lotes generados durante la iteración.

Iteramos sobre el dataloader utilizando un bucle "for" para obtener los lotes de datos y etiquetas.
La variable "datos_batch" corresponde al tensor que contiene las características del lote en esa iteración.
La variable "etiquetas_batch" corresponde al tensor que contiene las etiquetas asociadas al lote en esa iteración.
    
Aplicamos el método ".append()" a la lista "lotes_ordenados" para añadir cada lote obtenido del dataloader en cada iteración del bucle.
La tupla añadida contiene los tensores "datos_batch" y "etiquetas_batch" correspondientes al lote en cada iteración.

La variable "lotes_ordenados" contiene todos los lotes obtenidos del dataloader durante la iteración.
El batching permite dividir el dataset en mini-batches; el último puede ser más pequeño si no hay suficientes muestras."""

from torch.utils.data import DataLoader

dataloader_ordenado = DataLoader(dataset, batch_size=2, shuffle=False)

lotes_ordenados = []

for datos_batch, etiquetas_batch in dataloader_ordenado:
    lotes_ordenados.append((datos_batch, etiquetas_batch))

lotes_ordenados # Muestra todos los lotes obtenidos del dataloader ordenado en forma de lista de tuplas (datos, etiquetas). 


In [ ]:
# Bloque 3 — Batching con shuffling
"""Este bloque crea un dataloader con shuffling activado.
Aplicamos la clase "DataLoader()" a la variable "dataloader_barajado" para crear un dataloader a partir del dataset original.
El argumento "dataset" corresponde a la instancia del dataset original que se desea cargar en el dataloader.
El parámetro "batch_size=" especifica el tamaño máximo de los lotes que se obtendrán del dataloader.
El parámetro "shuffle=" indica si se debe mezclar el orden de las muestras antes de crear los lotes.
La variable "dataloader_barajado" almacena un objeto iterable que entrega los datos originales en lotes mezclados.

Creamos un contenedor de lista llamado "lotes_barajados" para almacenar los lotes obtenidos del dataloader.
La variable "lotes_barajados" almacena los lotes generados durante la iteración.

Iteramos sobre el dataloader utilizando un bucle "for" para obtener los lotes de datos y etiquetas.
La variable "datos_batch" corresponde al tensor que contiene las características del lote en esa iteración.
La variable "etiquetas_batch" corresponde al tensor que contiene las etiquetas asociadas al lote en esa iteración.
    
Aplicamos el método ".append()" a la lista "lotes_barajados" para añadir cada lote obtenido del dataloader en cada iteración del bucle.
La tupla añadida contiene los tensores "datos_batch" y "etiquetas_batch" correspondientes al lote en cada iteración.

La variable "lotes_barajados" contiene todos los lotes obtenidos del dataloader durante la iteración.
El shuffling puede variar el orden de los datos durante el entrenamiento, aunque por azar puede repetirlo."""

dataloader_barajado = DataLoader(dataset, batch_size=2, shuffle=True)

lotes_barajados = []

for datos_batch, etiquetas_batch in dataloader_barajado:
    lotes_barajados.append((datos_batch, etiquetas_batch))

lotes_barajados # Muestra todos los lotes obtenidos del dataloader barajado en forma de lista de tuplas (datos, etiquetas). 


In [ ]:
# Bloque 4 — Batching con tamaño de lote diferente
"""Este bloque crea un dataloader con un tamaño de lote distinto.
Aplicamos la clase "DataLoader()" a la variable "dataloader_grande" para crear un dataloader a partir del dataset original.
El argumento "dataset" corresponde a la instancia del dataset original que se desea cargar en el dataloader.
El parámetro "batch_size=" especifica el tamaño máximo de los lotes que se obtendrán del dataloader.
El parámetro "shuffle=" indica si se debe mezclar el orden de las muestras antes de crear los lotes.
La variable "dataloader_grande" almacena un objeto iterable que entrega los datos originales en lotes con un tamaño distinto al anterior.

Creamos un contenedor de lista llamado "lotes_grandes" para almacenar los lotes obtenidos del dataloader.
La variable "lotes_grandes" almacena los lotes generados durante la iteración.

Iteramos sobre el dataloader utilizando un bucle "for" para obtener los lotes de datos y etiquetas.
La variable "datos_batch" corresponde al tensor que contiene las características del lote en esa iteración.
La variable "etiquetas_batch" corresponde al tensor que contiene las etiquetas asociadas al lote en esa iteración.
    
Aplicamos el método ".append()" a la lista "lotes_grandes" para añadir cada lote obtenido del dataloader en cada iteración del bucle.
La tupla añadida contiene los tensores "datos_batch" y "etiquetas_batch" correspondientes al lote en cada iteración.

La variable "lotes_grandes" contiene todos los lotes obtenidos del dataloader durante la iteración.
El cambio en el tamaño de lote permite ajustar la carga de datos según el modelo."""

dataloader_grande = DataLoader(dataset, batch_size=3, shuffle=False)

lotes_grandes = []

for datos_batch, etiquetas_batch in dataloader_grande:
    lotes_grandes.append((datos_batch, etiquetas_batch))

lotes_grandes # Muestra todos los lotes obtenidos del dataloader grande en forma de lista de tuplas (datos, etiquetas).  


In [ ]:
# Bloque 5 — Comparación de órdenes
"""Este bloque compara el orden de los datos entre batching ordenado y batching con shuffling.
La variable "comparacion" almacena los lotes en ambos modos en forma de diccionario.
La comparación permite visualizar el efecto del shuffling sobre el flujo de datos."""

comparacion = {
    "ordenado": [(d.tolist(), e.tolist()) for d, e in lotes_ordenados],
    "barajado": [(d.tolist(), e.tolist()) for d, e in lotes_barajados]
}

comparacion # Muestra la comparación de los lotes entre el batching ordenado y el batching con shuffling.


In [ ]:
# Bloque 6 — Estructura final de resultados
"""Este bloque organiza los resultados principales del batching y shuffling.
La variable "resultado" almacena los lotes ordenados, barajados, grandes y la comparación en forma de diccionario.
La estructura permite visualizar el comportamiento del dataloader bajo diferentes configuraciones."""

resultado = {
    "lotes_ordenados": [(d.tolist(), e.tolist()) for d, e in lotes_ordenados],
    "lotes_barajados": [(d.tolist(), e.tolist()) for d, e in lotes_barajados],
    "lotes_grandes": [(d.tolist(), e.tolist()) for d, e in lotes_grandes],
    "comparacion": comparacion
}

resultado # Muestra la estructura final de resultados que incluye lotes ordenados, barajados, grandes y la comparación.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que el entrenamiento no depende solo del modelo, sino también de cómo se agrupan y ordenan los ejemplos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">El tamaño de lote y la mezcla de datos influyen en eficiencia, estabilidad y generalización.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Fijar batch size o activar shuffling por inercia sin observar su efecto real en el entrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Transformaciones para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Transformaciones y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">03_Transformaciones/01_Las_transformaciones.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">